### Loading libraries and initializing useful functions

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import (
    Button,
    IntSlider,
    VBox,
    fixed,
    interact,
    interactive_output,
)

%matplotlib inline

from config import IMAGE_DATA_DIR
from kuwahara_filters.classic import classic_kuwahara
from kuwahara_filters.generalized import (
    generalized_kuwahara,
    generate_sector_weights,
)

In [ ]:
def load_and_resize(image_path, max_side=800):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR_RGB)
    h, w = img.shape[:2]

    if max(h, w) <= max_side:
        return img

    scale = max_side / float(max(h, w))
    new_w = int(w * scale)
    new_h = int(h * scale)

    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

In [ ]:
def add_gaussian_noise(image, sigma):
    if sigma == 0:
        return image.copy()

    noise = np.random.normal(0, sigma, image.shape)
    noisy_image = image.astype(np.float32) + noise

    return np.clip(noisy_image, 0, 255).astype(np.uint8)

In [ ]:
def update_kuwahara(image, noise_sigma, kuwahara_algorithm, **kuwahara_params):
    noisy_image = add_gaussian_noise(image, sigma=noise_sigma)
    denoised_image = kuwahara_algorithm(noisy_image, **kuwahara_params)

    fig, axes = plt.subplots(1, 3, figsize=(15, 7))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(noisy_image)
    axes[1].set_title(f"Noisy (sigma={noise_sigma})")
    axes[1].axis("off")

    axes[2].imshow(denoised_image)
    params = ",".join(
        "=".join(map(str, pair)) for pair in kuwahara_params.items()
    )
    axes[2].set_title(f"Filtered ({params})")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

## Comparing generalized and classic filter

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "landscape.png", max_side=600)

interact(
    update_kuwahara,
    image=fixed(image),
    kuwahara_algorithm=fixed(generalized_kuwahara),
    noise_sigma=IntSlider(min=0, max=100, step=5, value=10),
    radius=IntSlider(min=1, max=10, step=1, value=1),
)

In [ ]:
noisy = add_gaussian_noise(image, sigma=10)
classic_filtered = classic_kuwahara(noisy, radius=5)
general_filtered = generalized_kuwahara(noisy, radius=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0][0].imshow(image)
axes[0][0].set_title("Original")

axes[0][1].imshow(noisy)
axes[0][1].set_title("Noisy")

axes[1][0].imshow(classic_filtered)
axes[1][0].set_title("Classic Kuwahara")

axes[1][1].imshow(general_filtered)
axes[1][1].set_title("General Kuwahara")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y, x = 150, 370
h, w = 75, 75

image_with_box = image.copy()
cv2.rectangle(image_with_box, (x, y), (x + w, y + h), 255, 2)

crop_classic = classic_filtered[y : y + h, x : x + w]
crop_general = general_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(1, 3, figsize=(12, 8))

axes[0].imshow(image_with_box)
axes[0].set_title("1. Classic filter", fontsize=14)

axes[1].imshow(crop_classic)
axes[1].set_title("Classic filter (Zoom)")

axes[2].imshow(crop_general)
axes[2].set_title("General filter (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y, x = 315, 370
h, w = 75, 75

classic_with_box = classic_filtered.copy()
cv2.rectangle(classic_with_box, (x, y), (x + w, y + h), 255, 2)

general_with_box = general_filtered.copy()
cv2.rectangle(general_with_box, (x, y), (x + w, y + h), 255, 2)

crop_classic = classic_filtered[y : y + h, x : x + w]
crop_general = general_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

axes[0][0].imshow(classic_with_box)
axes[0][0].set_title("1. Classic filter", fontsize=14)

axes[0][1].imshow(general_with_box)
axes[0][1].set_title("2. General filter", fontsize=14)

axes[1][0].imshow(crop_classic)
axes[1][0].set_title("Classic filter (Zoom)")

axes[1][1].imshow(crop_general)
axes[1][1].set_title("General filter (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
L = 31
sigma = 0.3 * ((L - 1) / 2 - 1) + 0.8

k1d = cv2.getGaussianKernel(L, sigma)
kernel = k1d @ k1d.T

H, W = image.shape[:2]
y, x = np.ogrid[-H // 2 : H // 2, -W // 2 : W // 2]

y_shifted, x_shifted = np.fft.ifftshift(y), np.fft.ifftshift(x)
gaussian_full = np.exp(-(x_shifted**2 + y_shifted**2) / (2 * sigma**2))
gaussian_full /= np.sum(gaussian_full)

gaussian_centered_for_display = np.fft.fftshift(gaussian_full)

fft_filter = np.fft.fft2(gaussian_full)
real_spectrum = np.real(np.fft.fftshift(fft_filter))

display(np.min(real_spectrum))
display(np.max(real_spectrum))

real_spectrum = np.clip(real_spectrum, a_min=0, a_max=None)

power = 0.3
amp_power_scaled = np.power(np.abs(real_spectrum), power)
signed_spectrum = amp_power_scaled * np.sign(real_spectrum)

vmax = np.max(signed_spectrum)
vmin = -vmax

fig, axes = plt.subplots(1, 2, figsize=(12, 12))

axes[0].imshow(gaussian_centered_for_display, cmap="gray")
axes[0].set_title("Gaussian Filter", fontsize=14)

axes[1].imshow(signed_spectrum, cmap="coolwarm", vmin=vmin, vmax=vmax)
axes[1].set_title("Filter Spectrum", fontsize=14)

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
size = 300
Y, X = np.ogrid[:size, :size]
center = size / 2
dist_sq = (X - center) ** 2 + (Y - center) ** 2

sigma = 50
clean_circle = np.exp(-dist_sq / (2 * sigma**2))
clean_circle = (clean_circle * 255).astype(np.float32)

np.random.seed(42)
noisy_circle = add_gaussian_noise(clean_circle, sigma=2)

clean_circle = clean_circle[..., np.newaxis]
noisy_circle = noisy_circle[..., np.newaxis]

radius = 5
filtered_circle_1 = classic_kuwahara(clean_circle, radius=radius)
filtered_noisy_circle_1 = classic_kuwahara(noisy_circle, radius=radius)

filtered_circle_2 = generalized_kuwahara(clean_circle, radius=radius)
filtered_noisy_circle_2 = generalized_kuwahara(noisy_circle, radius=radius)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].imshow(filtered_circle_1, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("1. Base Kuwahara (no noise)")

axes[0, 1].imshow(filtered_noisy_circle_1, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("2. Base Kuwahara (noise)")

axes[1, 0].imshow(filtered_circle_2, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("3. Generalized Kuwahara (no noise)")

axes[1, 1].imshow(filtered_noisy_circle_2, cmap="gray", vmin=0, vmax=255)
axes[1, 1].set_title("4. Generalized Kuwahara (noise)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

## Enhancing edges

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "mountain_iic.jpg", max_side=150)

upscaled_image = cv2.resize(
    image, dsize=None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC
)
general_filtered = generalized_kuwahara(upscaled_image, radius=9)

y, x = 200, 490
h, w = 100, 100

upscaled_with_box = upscaled_image.copy()
cv2.rectangle(upscaled_with_box, (x, y), (x + w, y + h), 255, 2)

general_with_box = general_filtered.copy()
cv2.rectangle(general_with_box, (x, y), (x + w, y + h), 255, 2)

crop_upscaled = upscaled_image[y : y + h, x : x + w]
crop_general = general_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].imshow(upscaled_with_box)
axes[0, 0].set_title("1. Upscaled image", fontsize=14)

axes[0, 1].imshow(general_with_box)
axes[0, 1].set_title("2. General filter", fontsize=14)

axes[1, 0].imshow(crop_upscaled)
axes[1, 0].set_title("Upscaled image (Zoom)")

axes[1, 1].imshow(crop_general)
axes[1, 1].set_title("General filter (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def calculate_kuwahara_weights(px, py):
    radius = 9
    N = 8
    q = 8.0

    window = upscaled_image[
        py - radius : py + radius + 1, px - radius : px + radius + 1
    ]
    window_float = window.astype(np.float32) / 255.0

    sector_kernels = generate_sector_weights(radius, N=N)
    stds = []
    sector_weights = []

    for kernel in sector_kernels:
        m = np.sum(window_float * kernel[:, :, np.newaxis], axis=(0, 1))
        m2 = np.sum(window_float**2 * kernel[:, :, np.newaxis], axis=(0, 1))
        s2 = np.clip(m2 - m**2, 0, None)
        s_total = np.sqrt(np.sum(s2)) + 1e-8

        stds.append(s_total)
        sector_weights.append(s_total ** (-q))

    normalized_weights = np.array(sector_weights) / np.sum(sector_weights)

    return {
        "window": window,
        "normalized_weights": normalized_weights,
        "stds": stds,
        "radius": radius,
        "N": N,
    }


def save_plots(data, px, py):
    radius = data["radius"]
    window = data["window"]
    normalized_weights = data["normalized_weights"]
    N = data["N"]

    fig1, ax1 = plt.subplots(figsize=(8, 6))
    ax1.imshow(upscaled_image)
    ax1.plot(px, py, "r+", markersize=15)
    ax1.axis("off")
    fig1.savefig(
        IMAGE_DATA_DIR / f"main_view_{px}_{py}.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=150,
    )
    plt.close(fig1)

    fig2 = plt.figure(figsize=(5, 5))
    ax2 = fig2.add_subplot(111, polar=True)
    theta_angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
    ax2.bar(
        theta_angles,
        normalized_weights,
        width=(2 * np.pi / N),
        alpha=0.7,
        edgecolor="k",
    )
    ax2.set_theta_direction(1)
    ax2.set_theta_offset(0)
    ax2.set_rlim(0, 1.0)
    ax2.set_rticks(np.arange(0.1, 1.1, 0.1))
    ax2.set_yticklabels([])
    ax2.set_rlabel_position(245)
    fig2.savefig(
        IMAGE_DATA_DIR / f"polar_weights_{px}_{py}.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=150,
    )
    plt.close(fig2)

    fig3, ax3 = plt.subplots(figsize=(4, 4))
    ax3.imshow(window)
    ax3.plot(radius, radius, "r+", markersize=15)
    ax3.axis("off")
    fig3.savefig(
        IMAGE_DATA_DIR / f"window_{px}_{py}.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=150,
    )
    plt.close(fig3)


def display_analysis(px, py):
    data = calculate_kuwahara_weights(px, py)
    radius = data["radius"]
    window = data["window"]
    normalized_weights = data["normalized_weights"]
    stds = data["stds"]
    N = data["N"]

    fig = plt.figure(figsize=(15, 5))

    ax1 = fig.add_subplot(131)
    ax1.imshow(upscaled_image)
    ax1.plot(px, py, "r+", markersize=15, label=f"Точка ({px}, {py})")
    ax1.axis("off")
    ax1.legend()

    theta_angles = np.linspace(0, 2 * np.pi, N, endpoint=False)

    ax2 = fig.add_subplot(132, polar=True)
    ax2.bar(
        theta_angles,
        normalized_weights,
        width=(2 * np.pi / N),
        alpha=0.7,
        edgecolor="k",
    )
    ax2.set_theta_direction(1)
    ax2.set_theta_offset(0)
    ax2.set_rlim(0, 1.0)
    ax2.set_rticks(np.arange(0.1, 1.1, 0.1))
    ax2.set_title("Веса секторов")

    ax3 = fig.add_subplot(133)
    ax3.imshow(window)
    ax3.plot(radius, radius, "r+", markersize=15)
    ax3.axis("off")

    plt.tight_layout()
    plt.show()

    for i, (weight, std) in enumerate(
        zip(normalized_weights, stds, strict=True)
    ):
        print(f"Сектор {i + 1}. Вес: {weight:.5f}, Ст. отклонение: {std:.5f}")


start_px, start_py = 575, 275

px_slider = IntSlider(
    min=start_px - 10,
    max=start_px + 10,
    step=1,
    value=start_px,
    description="X:",
)
py_slider = IntSlider(
    min=start_py - 10,
    max=start_py + 10,
    step=1,
    value=start_py,
    description="Y:",
)

save_button = Button(
    description="Сохранить изображения", button_style="success"
)


def on_save_button_clicked(_):
    px = px_slider.value
    py = py_slider.value

    data = calculate_kuwahara_weights(px, py)
    save_plots(data, px, py)


save_button.on_click(on_save_button_clicked)

interactive_plot = interactive_output(
    display_analysis, {"px": px_slider, "py": py_slider}
)

display(VBox([px_slider, py_slider, save_button]), interactive_plot)